# 01 — Exploratory Data Analysis
**Bank Retention Intelligence Platform**

This notebook covers:
- Dataset overview & validation
- Univariate distributions
- Churn by engagement, products, geography, age
- Correlation analysis
- Key findings summary

> **Data:** `data/raw/European_Bank.csv` — 10,000 European bank customers

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_cleaning import load_and_clean, get_summary
from src.visualization import (plot_churn_distribution, plot_feature_distributions,
                                plot_categorical_churn, plot_correlation_heatmap,
                                plot_engagement_vs_churn, plot_product_vs_churn)
from src.utils import save_processed, save_figure, get_path

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load & Validate Dataset

In [2]:
df = load_and_clean('../data/raw/European_Bank.csv')
summary = get_summary(df)

print("Dataset Summary")
print("=" * 40)
for k, v in summary.items():
    print(f"  {k:20s}: {v}")
print()
df.head()

Dataset Summary
  rows                : 10000
  columns             : 11
  churn_rate          : 20.37
  missing_values      : 0
  active_rate         : 51.51
  avg_balance         : 76369.72
  avg_products        : 1.53



,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Data Types & Missing Values

In [3]:
print("Dtypes:")
print(df.dtypes.to_string())
print(f"\nMissing values: {df.isna().sum().sum()}")
print("\nBasic statistics:")
df.describe().round(2)

Dtypes:
CreditScore          int32
Geography           object
Gender              object
Age                  int32
Tenure               int32
Balance            float64
NumOfProducts        int32
HasCrCard            int32
IsActiveMember       int32
EstimatedSalary    float64
Exited               int32

Missing values: 0

Basic statistics:


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00,10000.0
mean,650.74,38.90,5.01,76369.72,1.53,0.71,0.52,100089.58,0.2
std,96.12,10.32,2.89,62172.02,0.58,0.46,0.50,57477.44,0.4
min,432.00,21.00,0.00,0.00,1.00,0.00,0.00,1842.83,0.0
25%,584.00,32.00,3.00,0.00,1.00,0.00,0.00,51002.11,0.0
50%,652.00,37.00,5.00,97198.54,1.00,1.00,1.00,100193.92,0.0
75%,718.00,44.00,7.00,127644.24,2.00,1.00,1.00,149388.25,0.0
max,850.00,72.00,10.00,185967.99,4.00,1.00,1.00,198069.73,1.0


## 3. Target Variable — Churn Distribution

In [4]:
fig = plot_churn_distribution(df)
save_figure(fig, 'churn_distribution.png')
plt.show()
print(f"\nChurn rate  : {df['Exited'].mean()*100:.1f}%")
print(f"Retained    : {(df['Exited']==0).sum():,}")
print(f"Churned     : {(df['Exited']==1).sum():,}")

  Figure saved → E:\bank-retention-intelligence\outputs\figures\churn_distribution.png

Churn rate  : 20.4%
Retained    : 7,963
Churned     : 2,037


## 4. Numerical Feature Distributions

In [5]:
fig = plot_feature_distributions(df)
save_figure(fig, 'feature_distributions.png')
plt.show()

  Figure saved → E:\bank-retention-intelligence\outputs\figures\feature_distributions.png


## 5. Categorical Feature Churn Rates

In [6]:
fig = plot_categorical_churn(df)
save_figure(fig, 'categorical_churn_rates.png')
plt.show()

  Figure saved → E:\bank-retention-intelligence\outputs\figures\categorical_churn_rates.png


## 6. Engagement Analysis

In [7]:
active_churn   = df[df['IsActiveMember']==1]['Exited'].mean()*100
inactive_churn = df[df['IsActiveMember']==0]['Exited'].mean()*100
print(f"Active member churn   : {active_churn:.1f}%")
print(f"Inactive member churn : {inactive_churn:.1f}%")
print(f"Churn gap             : +{inactive_churn - active_churn:.1f} pp")

fig = plot_engagement_vs_churn(df)
save_figure(fig, 'engagement_vs_churn.png')
plt.show()

Active member churn   : 14.3%
Inactive member churn : 26.9%
Churn gap             : +12.6 pp


  Figure saved → E:\bank-retention-intelligence\outputs\figures\engagement_vs_churn.png


## 7. Product Utilisation Analysis

In [8]:
prod_churn = df.groupby('NumOfProducts')['Exited'].mean()*100
print("Churn rate by number of products:")
for p, r in prod_churn.items():
    bar = '█' * int(r/5)
    print(f"  {p} product(s): {r:.1f}%  {bar}")
print("\n→ Key insight: 2 products = lowest churn (7.6%)")

fig = plot_product_vs_churn(df)
save_figure(fig, 'products_vs_churn.png')
plt.show()

Churn rate by number of products:
  1 product(s): 27.7%  █████
  2 product(s): 7.6%  █
  3 product(s): 82.7%  ████████████████
  4 product(s): 100.0%  ████████████████████

→ Key insight: 2 products = lowest churn (7.6%)
  Figure saved → E:\bank-retention-intelligence\outputs\figures\products_vs_churn.png


## 8. Geography Analysis

In [9]:
geo_churn = df.groupby('Geography').agg(
    Count     =('Exited','count'),
    ChurnRate =('Exited', lambda x: round(x.mean()*100,1)),
    AvgBalance=('Balance','mean')
).reset_index()
print("Geography breakdown:")
print(geo_churn.to_string(index=False))

fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(geo_churn['Geography'], geo_churn['ChurnRate'],
              color=['#3266ad','#c0392b','#f0a500'], width=0.5, edgecolor='white')
for bar, val in zip(bars, geo_churn['ChurnRate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Rate (%)'); ax.set_title('Churn Rate by Geography', fontsize=13, fontweight='bold')
ax.set_ylim(0, 40); ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
save_figure(fig, 'geography_churn.png')
plt.show()

Geography breakdown:
Geography  Count  ChurnRate    AvgBalance
   France   5014       16.2  61981.269422
  Germany   2509       32.4 119656.405235
    Spain   2477       16.7  61649.253704


  Figure saved → E:\bank-retention-intelligence\outputs\figures\geography_churn.png


## 9. Age Group Analysis

In [10]:
df['AgeGroupTmp'] = pd.cut(df['Age'], bins=[0,25,35,45,55,120],
                              labels=['18-25','26-35','36-45','46-55','56+'])
age_churn = df.groupby('AgeGroupTmp', observed=True)['Exited'].mean()*100
print("Churn rate by age group:")
for ag, rate in age_churn.items():
    bar = '█' * int(rate/5)
    print(f"  {ag}: {rate:.1f}%  {bar}")
print("\n→ Key insight: Age 46-55 has 50.6% churn — highest of all groups")

fig, ax = plt.subplots(figsize=(7,4))
colors = ['#3266ad','#3266ad','#f0a500','#c0392b','#e67e22']
bars = ax.bar(age_churn.index, age_churn.values, color=colors, width=0.55, edgecolor='white')
for bar, val in zip(bars, age_churn.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Age Group'); ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Age Group', fontsize=13, fontweight='bold')
ax.set_ylim(0, 62); ax.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
save_figure(fig, 'age_group_churn.png')
plt.show()
df.drop(columns=['AgeGroupTmp'], inplace=True)

Churn rate by age group:
  18-25: 7.5%  █
  26-35: 8.5%  █
  36-45: 19.6%  ███
  46-55: 50.6%  ██████████
  56+: 36.8%  ███████

→ Key insight: Age 46-55 has 50.6% churn — highest of all groups
  Figure saved → E:\bank-retention-intelligence\outputs\figures\age_group_churn.png


## 10. Correlation Heatmap

In [11]:
fig = plot_correlation_heatmap(df)
save_figure(fig, 'correlation_heatmap.png')
plt.show()

  Figure saved → E:\bank-retention-intelligence\outputs\figures\correlation_heatmap.png


## 11. High-Value Disengaged Customers

In [12]:
median_bal = df['Balance'].median()
hv = df[(df['Balance'] > median_bal) & (df['IsActiveMember'] == 0)]
hv_churn = hv['Exited'].mean()*100

print("High-Value Disengaged Customers")
print("=" * 42)
print(f"  Definition : Balance > €{median_bal:,.0f} AND IsActiveMember=0")
print(f"  Count      : {len(hv):,} ({len(hv)/len(df)*100:.1f}% of base)")
print(f"  Churn rate : {hv_churn:.1f}%  (vs {df['Exited'].mean()*100:.1f}% overall)")
print(f"  Avg balance: €{hv['Balance'].mean():,.0f}")
print(f"  Revenue @ risk: €{hv[hv['Exited']==1]['Balance'].sum()/1e6:.1f}M")

High-Value Disengaged Customers
  Definition : Balance > €97,199 AND IsActiveMember=0
  Count      : 2,456 (24.6% of base)
  Churn rate : 32.3%  (vs 20.4% overall)
  Avg balance: €130,926
  Revenue @ risk: €103.6M


## 12. Save Cleaned Dataset

In [13]:
save_processed(df, 'cleaned_dataset.csv')
print("\nEDA complete. Key findings:")
print("  1. Inactive members churn at 26.9% vs 14.3% for active (12.6 pp gap)")
print("  2. 2-product customers churn at only 7.6% — single-product at 27.7%")
print("  3. Germany churns at 32.4% — nearly double France (16.2%)")
print("  4. Age 46-55 has 50.6% churn — highest cohort")
print("  5. 2,456 high-value disengaged customers — €104.7M at risk")
print("\nNext: Run 02_preprocessing.ipynb")

  Saved → E:\bank-retention-intelligence\data\processed\cleaned_dataset.csv  (10,000 rows)

EDA complete. Key findings:
  1. Inactive members churn at 26.9% vs 14.3% for active (12.6 pp gap)
  2. 2-product customers churn at only 7.6% — single-product at 27.7%
  3. Germany churns at 32.4% — nearly double France (16.2%)
  4. Age 46-55 has 50.6% churn — highest cohort
  5. 2,456 high-value disengaged customers — €104.7M at risk

Next: Run 02_preprocessing.ipynb
